To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth your local device, follow [our guide](https://docs.unsloth.ai/get-started/install-and-update). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)


### News


Introducing FP8 precision training for faster RL inference. [Read Blog](https://docs.unsloth.ai/new/fp8-reinforcement-learning).

Unsloth's [Docker image](https://hub.docker.com/r/unsloth/unsloth) is here! Start training with no setup & environment issues. [Read our Guide](https://docs.unsloth.ai/new/how-to-train-llms-with-unsloth-and-docker).

[gpt-oss RL](https://docs.unsloth.ai/new/gpt-oss-reinforcement-learning) is now supported with the fastest inference & lowest VRAM. Try our [new notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/gpt-oss-(20B)-GRPO.ipynb) which creates kernels!

Introducing [Vision](https://docs.unsloth.ai/new/vision-reinforcement-learning-vlm-rl) and [Standby](https://docs.unsloth.ai/basics/memory-efficient-rl) for RL! Train Qwen, Gemma etc. VLMs with GSPO - even faster with less VRAM.

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


### Installation

In [1]:
import torch

import bitsandbytes as bnb



print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Version: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")


PyTorch Version: 2.9.0+cu128
CUDA Version: 12.8
GPU: NVIDIA GeForce RTX 5070 Ti


In [2]:
# %%capture
import os
import re

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch

    v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + (
        "0.0.33.post1"
        if v == "2.9"
        else "0.0.32.post2"
        if v == "2.8"
        else "0.0.29.post3"
    )
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Unsloth


In [3]:
import os

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

from unsloth import FastLanguageModel
import torch

fourbit_models = [
    "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit",  # Qwen 14B 2x faster
    "unsloth/Qwen3-4B-Thinking-2507-unsloth-bnb-4bit",
    "unsloth/Qwen3-8B-unsloth-bnb-4bit",
    "unsloth/Qwen3-14B-unsloth-bnb-4bit",
    "unsloth/Qwen3-32B-unsloth-bnb-4bit",
    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/DeepSeek-R1-0528-Qwen3-8B-unsloth-bnb-4bit",
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/Phi-4",
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/orpheus-3b-0.1-ft-unsloth-bnb-4bit",  # [NEW] We support TTS models!
]  # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit",
    max_seq_length=2048,  # Choose any for long context!
    load_in_4bit=True,  # 4 bit quantization to reduce memory
    load_in_8bit=False,  # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning=False,  # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


TMA benchmarks will be running without grid constant TMA descriptor.


WARNING 12-19 06:59:54 [interface.py:409] Using 'pin_memory=False' as WSL is detected. This may slow down the performance.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: FBGEMM on the current GPU cannot load - will switch to Triton kernels


[unsloth_zoo.log|WARNING]Unsloth: Failed to import trl openenv: No module named 'trl.experimental'


==((====))==  Unsloth 2025.12.4: Fast Qwen3 patching. Transformers: 4.56.2. vLLM: 0.11.2.
   \\   /|    NVIDIA GeForce RTX 5070 Ti. Num GPUs = 1. Max memory: 15.92 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


We now add LoRA adapters so we only need to update a small amount of parameters!

In [4]:

model = FastLanguageModel.get_peft_model(
    model,
    r=32,  # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,  # Supports any, but = 0 is optimized
    bias="none",  # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing="unsloth",  # True or "unsloth" for very long context
    random_state=3407,
    use_rslora=False,  # We support rank stabilized LoRA
    loftq_config=None,  # And LoftQ
)

Unsloth 2025.12.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


<a name="Data"></a>
### Data Prep


```
<|im_start|>user
Hello!<|im_end|>
<|im_start|>assistant
Hey there!<|im_end|>

```
We use our `get_chat_template` function to get the correct chat template. We support `zephyr, chatml, mistral, llama, alpaca, vicuna, vicuna_old, phi3, llama3, phi4, qwen2.5, gemma3` and more.

In [19]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen3-instruct",
)

In [20]:
from datasets import load_dataset

dataset = load_dataset(
    "json", data_files="work/sft_data_qwen_format.jsonl", split="train"
)

Generating train split: 0 examples [00:00, ? examples/s]

We now use `standardize_data_formats` to try converting datasets to the correct format for finetuning purposes!

In [21]:
from unsloth.chat_templates import standardize_data_formats

dataset = standardize_data_formats(dataset)

Unsloth: Standardizing formats (num_proc=32):   0%|          | 0/10995 [00:00<?, ? examples/s]

Let's see how row 100 looks like!

In [22]:
print(len(dataset))

10995


In [24]:
dataset[2343]

{'conversations': [{'role': 'system',
   'content': '你是一位专业的教师。你的任务是根据给定的教学目标和知识点，采用启发式教学法引导学生进行思考。\n\n### 教学背景与目标\n教学目标：1. 通过创新电路设计和程序控制，掌握自动化装置的基本原理与构造方法，培养解决真实问题的能力。2. 通过工程制作实践活动，发展学生的工程思维、动手能力和跨学科综合素养。\n\n### 当前学生画像\n类型：知识掌握不足\n场景描述：（2）学生不理解教师讲解的内容\n\n### 教学要求\n1. 不要直接给出答案，通过提问引导学生。\n2. 结合学生的回答情况进行追问或评价。\n3. 最终引导学生达成教学目标。'},
  {'role': 'user', 'content': '你知道自动化装置中的传感器起什么作用吗？'},
  {'role': 'assistant', 'content': '这个问题涉及到了传感器的基本定义和功能，首先，请告诉我，你知道什么是传感器吗？'},
  {'role': 'user',
   'content': '传感器是一种能够感知外界环境中的各种物理量、化学量或生物量，并将其转换为电信号或其他形式的信号输出的装置。'},
  {'role': 'assistant', 'content': '很好，那么在自动化装置中，你知道传感器起什么作用吗？'},
  {'role': 'user',
   'content': '自动化装置中的传感器起着监测和反馈的作用，例如在智能温控系统中，温度传感器会实时监测环境温度，并将数据反馈给控制系统，从而调节空调或加热设备的工作状态。'},
  {'role': 'assistant', 'content': '那么，你能否举出一些传感器在不同应用场景中的具体应用呢？'},
  {'role': 'user', 'content': '好的，我们来一起想想看。比如在工业生产中，传感器可以用来监测什么？'},
  {'role': 'assistant', 'content': '在工业生产中，传感器可以用来监测什么？'},
  {'role': 'user',
   'content': '在工业生产中，传感器的应用非常广泛。例如，温度传感器可以用来监测炉温，

We now have to apply the chat template for `Qwen-3` onto the conversations, and save it to `text`.

In [25]:
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        )
        for convo in convos
    ]
    return {
        "text": texts,
    }


dataset = dataset.map(formatting_prompts_func, batched=True)

Map:   0%|          | 0/10995 [00:00<?, ? examples/s]

Let's see how the chat template did!


In [26]:
dataset[23]["text"]

"<|im_start|>system\n你是一位专业的教师。你的任务是根据给定的教学目标和知识点，采用启发式教学法引导学生进行思考。\n\n### 教学背景与目标\n教学目标：1. 模拟沉船过程，分析沉船原因，理解浮力变化与沉浮状态的关联，培养学生理论联系实际的能力。2. 运用物体沉浮条件对轮船受力进行分析，增强受力图绘制与科学建模能力。3. 通过分析沉船打捞方案，发展学生的科学态度、逻辑推理与工程思维，强化学以致用意识。\n\n### 当前学生画像\n类型：知识掌握不足\n场景描述：（4）学生求知欲弱\n\n### 教学要求\n1. 不要直接给出答案，通过提问引导学生。\n2. 结合学生的回答情况进行追问或评价。\n3. 最终引导学生达成教学目标。<|im_end|>\n<|im_start|>user\nEffect of water pressure on the buoyancy of an object\n\nIn the context of the ship's sunken condition, how does the water pressure affect the buoyant force?\n\nTo understand the effect of water pressure on the buoyant force, can we consider the ship as a whole and analyze how varying water pressure might influence the overall buoyancy?<|im_end|>\n<|im_start|>assistant\nwater pressure increases with depth, correct? So, when a ship is sunken, the water pressure on its lower part will be greater than on its upper part. This difference in pressure will cause a net force upwards, increasing the buoyant force. Is this understanding corre

In [27]:
print(len(dataset))

10995


<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [28]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    eval_dataset=None,  # Can set up evaluation!
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,  # Use GA to mimic batch size!
        warmup_steps=5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps=60,
        learning_rate=2e-4,  # Reduce to 2e-5 for long training runs
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=3407,
        report_to="none",  # Use TrackIO/WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=36):   0%|          | 0/10995 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. This helps increase accuracy of finetunes!

In [29]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

Map (num_proc=36):   0%|          | 0/10995 [00:00<?, ? examples/s]

Let's verify masking the instruction part is done! Let's print the 100th row again.

In [30]:
tokenizer.decode(trainer.train_dataset[2223]["input_ids"])

'<|im_start|>system\n你是一位专业的教师。你的任务是根据给定的教学目标和知识点，采用启发式教学法引导学生进行思考。\n\n### 教学背景与目标\n认识东南亚的世界遗产\n\n### 当前学生画像\n类型：学习渴望低\n场景描述：（2）学生不理解教师讲解的内容\n\n### 教学要求\n1. 不要直接给出答案，通过提问引导学生。\n2. 结合学生的回答情况进行追问或评价。\n3. 最终引导学生达成教学目标。<|im_end|>\n<|im_start|>user\n东南亚的世界遗产有哪些具体的分布情况和保护措施？<|im_end|>\n<|im_start|>assistant\n这个问题非常好，我们可以先来看一下吴哥窟，它位于柬埔寨，是世界文化遗产之一。那么，你知道吴哥窟有哪些保护措施吗？<|im_end|>\n<|im_start|>user\n吴哥窟的保护措施主要包括：限制游客数量以减少对遗址的损害，定期进行文物修复和维护，加强防洪排水系统以防止洪水侵蚀，以及开展国际间的合作研究和保护项目。<|im_end|>\n<|im_start|>assistant\n很好，吴哥窟的保护措施确实非常全面。那么，除了吴哥窟之外，你还知道东南亚有哪些其他世界遗产？<|im_end|>\n<|im_start|>user\n除了吴哥窟，东南亚还有许多其他世界遗产。例如，泰国的大城府，那里有丰富的历史遗迹；印度尼西亚的婆罗浮屠，是世界上最大的佛教石雕建筑；还有越南的下龙湾，以其独特的喀斯特地貌闻名。这些地方都有各自独特的文化和自然景观。<|im_end|>\n<|im_start|>assistant\n看来你对东南亚世界遗产的分布情况非常熟悉。那么，除了分布情况，你对这些遗产的保护措施了解多少呢？<|im_end|>\n<|im_start|>user\n泰国的大城府遗址保护措施包括限制游客数量、修复古迹和保护环境。婆罗浮屠则通过限制游客数量、修复和维护建筑、以及加强防洪排水系统来保护。下龙湾则通过限制游客数量、保护自然环境、以及开展环保教育来保护。<|im_end|>\n<|im_start|>assistant\n非常好，你对东南亚世界遗产的保护措施了解得很全面。那么，我们可以试着总结一下这些保护措施的共同点，你认为它们有什么相似之处呢？<|im_en

In [31]:
print(trainer.train_dataset[2223]["labels"])


[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 105073, 108014, 3837, 105773, 60726, 100192, 100158, 103948, 99826, 112655, 3837, 99652, 103987, 116388, 3837, 20412, 99489, 108077, 100653, 1773, 100624, 3837, 107733, 103948, 99826, 112655, 104719, 100153, 101082, 101037, 11319, 

Now let's print the masked out example - you should see only the answer is present:

In [32]:
tokenizer.decode([
    tokenizer.pad_token_id if x == -100 else x
    for x in trainer.train_dataset[23]["labels"]
]).replace(tokenizer.pad_token, " ")

"                                                                                                                                                                                                                                                                             water pressure increases with depth, correct? So, when a ship is sunken, the water pressure on its lower part will be greater than on its upper part. This difference in pressure will cause a net force upwards, increasing the buoyant force. Is this understanding correct?<|im_end|>\n                                                               Yes, that's correct! The difference in water pressure indeed causes a net force upwards, increasing the buoyant force when a ship is sunken. Now, can you explain how this increased buoyant force affects the ship's sunken condition?<|im_end|>\n"

In [33]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA GeForce RTX 5070 Ti. Max memory = 15.92 GB.
3.949 GB of memory reserved.


Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [34]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,995 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 66,060,288 of 4,088,528,384 (1.62% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,2.646600
2,3.364000
3,2.710100
4,3.204700
5,3.369300
6,2.471000
7,1.717000
8,1.313300
9,1.255800
10,1.434100


In [35]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime'] / 60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

247.7024 seconds used for training.
4.13 minutes used for training.
Peak reserved memory = 9.283 GB.
Peak reserved memory for training = 5.334 GB.
Peak reserved memory % of max memory = 58.31 %.
Peak reserved memory for training % of max memory = 33.505 %.


<a name="Inference"></a>
### Inference
Let's run the model via Unsloth native inference! According to the `Qwen-3` team, the recommended settings for instruct inference are `temperature = 0.7, top_p = 0.8, top_k = 20`

For reasoning chat based inference, `temperature = 0.6, top_p = 0.95, top_k = 20`

In [36]:
messages =  [{'role': 'system', 'content': '你是一位专业的生物学科教师。你的任务是根据给定的教学目标和知识点，采用启发式教学法引导学生进行思考。\n\n### 教学背景与目标\n教学目标：1. 观察并分析厨房烹饪过程中的物态变化现象，提出科学改进建议并进行实验验证。2. 培养学生科学探究能力、跨学科综合运用知识的能力以及安全意识和实践精神。\n\n### 当前学生画像\n类型：学习渴望低\n场景描述：（5）学生求知欲弱\n\n### 教学要求\n1. 不要直接给出答案，通过提问引导学生。\n2. 结合学生的回答情况进行追问或评价。\n3. 最终引导学生达成教学目标。'}, {'role': 'user', 'content': '为什么水在煮沸时会产生气泡，而油在高温下却不会像水一样沸腾产生气泡？'}]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,  # Must add for generation
)

from transformers import TextStreamer

_ = model.generate(
    **tokenizer(text, return_tensors="pt").to("cuda"),
    max_new_tokens=1000,  # Increase for longer outputs!
    temperature=0.7,
    top_p=0.8,
    top_k=20,  # For non thinking
    streamer=TextStreamer(tokenizer, skip_prompt=True),
)

这个问题涉及到了物质在不同状态下的物理性质。水和油都是液体，但它们的分子结构和分子间作用力不同。当水被加热时，水分子获得足够的能量，克服了分子间的吸引力，从液态转变为气态，形成水蒸气，从而在容器底部和侧壁上产生气泡。那么，油在高温下为什么会保持液态，而不是像水一样沸腾产生气泡呢？<|im_end|>


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [38]:
model.save_pretrained("work/lora_model")  # Local saving
tokenizer.save_pretrained("worl/lora_model")
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/lora_model", token = "...") # Online saving

('worl/lora_model/tokenizer_config.json',
 'worl/lora_model/special_tokens_map.json',
 'worl/lora_model/chat_template.jinja',
 'worl/lora_model/vocab.json',
 'worl/lora_model/merges.txt',
 'worl/lora_model/added_tokens.json',
 'worl/lora_model/tokenizer.json')

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="lora_model",  # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length=2048,
        load_in_4bit=True,
    )

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Merge to 16bit
if False:
    model.save_pretrained_merged(
        "model",
        tokenizer,
        save_method="merged_16bit",
    )
if False:  # Pushing to HF Hub
    model.push_to_hub_merged(
        "hf/model", tokenizer, save_method="merged_16bit", token=""
    )

# Merge to 4bit
if False:
    model.save_pretrained_merged(
        "model",
        tokenizer,
        save_method="merged_4bit",
    )
if False:  # Pushing to HF Hub
    model.push_to_hub_merged("hf/model", tokenizer, save_method="merged_4bit", token="")

# Just LoRA adapters
if False:
    model.save_pretrained("model")
    tokenizer.save_pretrained("model")
if False:  # Pushing to HF Hub
    model.push_to_hub("hf/model", token="")
    tokenizer.push_to_hub("hf/model", token="")


### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

Likewise, if you want to instead push to GGUF to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [ ]:
# Save to 8bit Q8_0
if False:
    model.save_pretrained_gguf(
        "model",
        tokenizer,
    )
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False:
    model.push_to_hub_gguf("hf/model", tokenizer, token="")

# Save to 16bit GGUF
if False:
    model.save_pretrained_gguf("model", tokenizer, quantization_method="f16")
if False:  # Pushing to HF Hub
    model.push_to_hub_gguf("hf/model", tokenizer, quantization_method="f16", token="")

# Save to q4_k_m GGUF
if False:
    model.save_pretrained_gguf("model", tokenizer, quantization_method="q4_k_m")
if False:  # Pushing to HF Hub
    model.push_to_hub_gguf(
        "hf/model", tokenizer, quantization_method="q4_k_m", token=""
    )

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "hf/model",  # Change hf to your username!
        tokenizer,
        quantization_method=[
            "q4_k_m",
            "q8_0",
            "q5_k_m",
        ],
        token="",  # Get a token at https://huggingface.co/settings/tokens
    )

Now, use the `model-unsloth.gguf` file or `model-unsloth-Q4_K_M.gguf` file in llama.cpp.

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).
